# Humanitarian data analysis via humanitarian-mcp

Four reproducible research workflows against a local `humanitarian-mcp` server.
Start it first: `npm run dashboard` (see `examples/notebooks/README.md`).

Every extraction below carries a **manifest** (exact arguments, timestamp, server
version, citation) — paste it into your paper's appendix and anyone can reproduce the pull.

In [ ]:
import io
import pandas as pd
import requests

SERVER = "http://localhost:8642"


def call(tool: str, **arguments):
    """One MCP tool call via the dashboard bridge; returns structuredContent."""
    response = requests.post(f"{SERVER}/api/call", json={"tool": tool, "arguments": arguments}, timeout=120)
    response.raise_for_status()
    payload = response.json()
    if payload.get("isError"):
        raise RuntimeError(payload["content"][0]["text"])
    return payload["structuredContent"]


def export_csv(**arguments) -> pd.DataFrame:
    """export_data → DataFrame. The manifest rides in '#' comment lines."""
    result = call("export_data", format="csv", **arguments)
    return pd.read_csv(io.StringIO(result["data"]), comment="#")


call("provider_health")["providers"]

## Workflow 1 — Event study: what did the April 2023 war do to displacement from Sudan?

First let the server's anomaly detection point at the break, then fit an
interrupted time-series (ITS) regression on the exported series.

In [ ]:
trend = call("trend_analysis", country="Sudan", role="origin", year_from=2010)
trend["anomalies"]

In [ ]:
import statsmodels.formula.api as smf

sudan = export_csv(dataset="population", country="Sudan", role="origin", year_from=2010)
yearly = sudan.groupby("year", as_index=False)["refugees"].sum()
yearly["post"] = (yearly.year >= 2023).astype(int)
yearly["t"] = yearly.year - yearly.year.min()

model = smf.ols("refugees ~ t + post + t:post", data=yearly).fit()
print(model.summary().tables[1])

## Workflow 2 — Asylum policy: recognition rates across host states

`recognition_rate_pct` — the standard dependent variable in asylum-policy
literature — is computed per year with the formula stated in the output.

In [ ]:
frames = []
for country in ["Egypt", "Germany", "Kenya"]:
    decisions = call("asylum_decisions", country=country, year_from=2019)
    df = pd.DataFrame(decisions["yearly"])
    df["country"] = country
    frames.append(df)

rates = pd.concat(frames)
rates.pivot_table(index="year", columns="country", values="recognition_rate_pct")

## Workflow 3 — Hosting burden per capita

Absolute numbers hide the burden: per 1,000 residents, Lebanon, Jordan and Chad
lead instead of large economies. Denominators are matched **per year** from the
World Bank provider, and the denominator year is disclosed per row.

In [ ]:
import matplotlib.pyplot as plt

ranking = call("top_host_countries", normalize_by="population", limit=10)
per_capita = pd.DataFrame(ranking["ranking"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(per_capita.country[::-1], per_capita.value[::-1])
ax.set_xlabel(f"refugees {ranking['unit']} ({ranking['year']})")
ax.set_title("Refugee hosting burden, per capita")
plt.tight_layout()
per_capita[["rank", "country", "value", "raw_value", "denominator_year"]]

## Workflow 4 — Conflict × displacement (needs the `hdx` provider)

Relate ACLED conflict intensity to displacement from the same country.
Start the server with `HMCP_PROVIDERS=unhcr,worldbank,hdx` and a free
`HMCP_HDX_APP_ID` (see `.env.example`); this cell explains itself otherwise.

In [ ]:
try:
    conflict = pd.DataFrame(call("conflict_events", country="Sudan", year_from=2018)["records"])
    displaced = export_csv(dataset="population", country="Sudan", role="origin", year_from=2018)
    displaced_yearly = displaced.groupby("year", as_index=False)["refugees"].sum()
    merged = conflict.merge(displaced_yearly, on="year")
    print(merged)
    print("\ncorr(fatalities, refugees) =", merged.fatalities.corr(merged.refugees).round(3))
except RuntimeError as err:
    print("Skipped — enable the hdx provider to run this workflow:\n", err)

## Method notes

- Figures are end-year stocks; UNHCR revises series retroactively — **record your
  extraction date** (the manifest does this for you).
- `role="asylum"` = hosted *in* the country; `role="origin"` = displaced *from* it.
- Add `include_codebook=True` to any `export_data` call for a variable-level
  codebook ready for your data appendix.
- Cite the data as UNHCR / World Bank / the original HDX producers (ACLED, IPC,
  OCHA FTS, IOM DTM), and the tooling via the repository's CITATION.cff.